Since you're on Windows PowerShell and using:

tnc -p 443 8.8.8.8


(tnc = Test-NetConnection)

you can execute the same command from Python using the subprocess module.

Option 1: Run Test-NetConnection from Python
import subprocess

ip = "8.8.8.8"
port = 443

result = subprocess.run(
    ["powershell", "-Command", f"Test-NetConnection {ip} -Port {port}"],
    capture_output=True,
    text=True
)

print(result.stdout)

Option 2: Check only whether the TCP test succeeded
import subprocess

ip = "8.8.8.8"
port = 443

cmd = f"(Test-NetConnection {ip} -Port {port).TcpTestSucceeded"

result = subprocess.run(
    ["powershell", "-Command", cmd],
    capture_output=True,
    text=True
)

if result.stdout.strip() == "True":
    print(f"TCP connection to {ip}:{port} successful")
else:
    print(f"TCP connection to {ip}:{port} failed")

Option 3: Native Python (Recommended)

Instead of launching PowerShell, you can test TCP connectivity directly with Python:

import socket

ip = "8.8.8.8"
port = 443

try:
    sock = socket.create_connection((ip, port), timeout=5)
    print(f"Connected to {ip}:{port}")
    sock.close()
except Exception as e:
    print(f"Connection failed: {e}")


Output:

Connected to 8.8.8.8:443


This is essentially the Python equivalent of:

Test-NetConnection 8.8.8.8 -Port 443

Option 4: ICMP Ping (equivalent to ping 8.8.8.8)
import subprocess

result = subprocess.run(
    ["ping", "-n", "4", "8.8.8.8"],
    capture_output=True,
    text=True
)

print(result.stdout)


Difference:

ping → ICMP reachability test.
Test-NetConnection -Port 443 → TCP connectivity test.
socket.create_connection() → Native Python TCP test (best for network automation scripts).

For network automation, you'll often want to parse:

Packet loss %
Min/Avg/Max RTT
Whether the host is reachable
Example Windows Ping Output
Pinging 8.8.8.8 with 32 bytes of data:
Reply from 8.8.8.8: bytes=32 time=12ms TTL=117
Reply from 8.8.8.8: bytes=32 time=11ms TTL=117
Reply from 8.8.8.8: bytes=32 time=10ms TTL=117
Reply from 8.8.8.8: bytes=32 time=13ms TTL=117

Ping statistics for 8.8.8.8:
    Packets: Sent = 4, Received = 4, Lost = 0 (0% loss),
Approximate round trip times in milli-seconds:
    Minimum = 10ms, Maximum = 13ms, Average = 11ms

Parse Packet Loss and RTT
import subprocess
import re

ip = "8.8.8.8"

result = subprocess.run(
    ["ping", "-n", "4", ip],
    capture_output=True,
    text=True
)

output = result.stdout

print(output)

# Packet statistics
packet_match = re.search(
    r"Sent = (\d+), Received = (\d+), Lost = (\d+) \((\d+)% loss\)",
    output
)

if packet_match:
    sent = int(packet_match.group(1))
    received = int(packet_match.group(2))
    lost = int(packet_match.group(3))
    loss_pct = int(packet_match.group(4))

    print(f"Sent: {sent}")
    print(f"Received: {received}")
    print(f"Lost: {lost}")
    print(f"Loss: {loss_pct}%")

# RTT statistics
rtt_match = re.search(
    r"Minimum = (\d+)ms, Maximum = (\d+)ms, Average = (\d+)ms",
    output
)

if rtt_match:
    min_rtt = int(rtt_match.group(1))
    max_rtt = int(rtt_match.group(2))
    avg_rtt = int(rtt_match.group(3))

    print(f"Min RTT: {min_rtt} ms")
    print(f"Max RTT: {max_rtt} ms")
    print(f"Avg RTT: {avg_rtt} ms")

Store Results in a Dictionary

This is common in automation scripts:

import subprocess
import re

def ping_host(ip):
    result = subprocess.run(
        ["ping", "-n", "4", ip],
        capture_output=True,
        text=True
    )

    output = result.stdout

    stats = {}

    packet = re.search(
        r"Sent = (\d+), Received = (\d+), Lost = (\d+) \((\d+)% loss\)",
        output
    )

    rtt = re.search(
        r"Minimum = (\d+)ms, Maximum = (\d+)ms, Average = (\d+)ms",
        output
    )

    if packet:
        stats["sent"] = int(packet.group(1))
        stats["received"] = int(packet.group(2))
        stats["lost"] = int(packet.group(3))
        stats["loss_percent"] = int(packet.group(4))

    if rtt:
        stats["min_rtt"] = int(rtt.group(1))
        stats["max_rtt"] = int(rtt.group(2))
        stats["avg_rtt"] = int(rtt.group(3))

    return stats

print(ping_host("8.8.8.8"))


Output:

{
    'sent': 4,
    'received': 4,
    'lost': 0,
    'loss_percent': 0,
    'min_rtt': 10,
    'max_rtt': 13,
    'avg_rtt': 11
}

Network-Automation Style Reachability Check
import subprocess

ip = "8.8.8.8"

result = subprocess.run(
    ["ping", "-n", "1", ip],
    capture_output=True,
    text=True
)

if "TTL=" in result.stdout:
    print(f"{ip} is reachable")
else:
    print(f"{ip} is unreachable")


This is the style frequently used in WAN automation, health checks, and lab validation scripts before running BGP, IS-IS, or traffic tests.

In [2]:
import subprocess

ip = "8.8.8.8"
port = 443

result = subprocess.run(
    ["powershell", "-Command", f"Test-NetConnection {ip} -Port {port}"],
    capture_output=True,
    text=True
)

print(result.stdout)



ComputerName     : 8.8.8.8
RemoteAddress    : 8.8.8.8
RemotePort       : 443
InterfaceAlias   : Ethernet 2
SourceAddress    : 10.0.9.200
TcpTestSucceeded : True






In [5]:
import subprocess

ip = "8.8.8.8"
port = 443

cmd = (f"Test-NetConnection {ip} -Port {port}.TcpTestSucceeded")

result = subprocess.run(
    ["powershell", "-Command", cmd],
    capture_output=True,
    text=True
)

if result.stdout.strip() == "True":
    print(f"TCP connection to {ip}:{port} successful")
else:
    print(f"TCP connection to {ip}:{port} failed")

TCP connection to 8.8.8.8:443 failed


In [18]:
import subprocess

ip = "8.8.8.8"
port = 443

cmd = (f"Test-NetConnection {ip} -Port {port}")

result = subprocess.run(
    ["powershell", "-Command", cmd],
    capture_output=True,
    text=True
)

if result.stdout.strip() == "True":
    print(f"TCP connection to {ip}:{port} successful")
else:
    print(f"TCP connection to {ip}:{port} failed")

TCP connection to 8.8.8.8:443 failed


In [20]:
import subprocess

ip = "8.8.8.8"
port = 443

cmd = (f"Test-NetConnection {ip} -Port {port} -InformationLevel Quiet")

result = subprocess.run(
    ["powershell", "-Command", cmd],
    capture_output=True,
    text=True
)

if result.stdout.strip() == "True":
    print(f"TCP connection to {ip}:{port} successful")
else:
    print(f"TCP connection to {ip}:{port} failed")

TCP connection to 8.8.8.8:443 successful


In [19]:
import subprocess

ip = "8.8.8.8"
port = 443

cmd = f"Test-NetConnection {ip} -Port {port} -InformationLevel Quiet"

result = subprocess.run(
    ["powershell", "-Command", cmd],
    capture_output=True,
    text=True
)

print("PowerShell returned:", result.stdout.strip())

if result.stdout.strip() == "True":
    print(f"TCP connection to {ip}:{port} successful")
else:
    print(f"TCP connection to {ip}:{port} failed")

PowerShell returned: True
TCP connection to 8.8.8.8:443 successful


In [21]:
import subprocess

ip = "8.8.8.8"
port = 443

cmd = f"Test-NetConnection {ip} -Port {port}"

result = subprocess.run(
    ["powershell", "-Command", cmd],
    capture_output=True,
    text=True
)

print(result.stdout)

if "TcpTestSucceeded : True" in result.stdout:
    print(f"TCP connection to {ip}:{port} successful")
else:
    print(f"TCP connection to {ip}:{port} failed")



ComputerName     : 8.8.8.8
RemoteAddress    : 8.8.8.8
RemotePort       : 443
InterfaceAlias   : Ethernet 2
SourceAddress    : 10.0.9.200
TcpTestSucceeded : True




TCP connection to 8.8.8.8:443 successful


In [22]:
import socket

ip = "8.8.8.8"
port = 443

try:
    with socket.create_connection((ip, port), timeout=5):
        print(f"TCP connection to {ip}:{port} successful")

except (socket.timeout, OSError) as error:
    print(f"TCP connection to {ip}:{port} failed")
    print("Reason:", error)

TCP connection to 8.8.8.8:443 successful


In [6]:
import socket

ip = "8.8.8.8"
port = 443

try:
    sock = socket.create_connection((ip, port), timeout=5)
    print(f"Connected to {ip}:{port}")
    sock.close()
except Exception as e:
    print(f"Connection failed: {e}")


Connected to 8.8.8.8:443


In [7]:
import subprocess

result = subprocess.run(
    ["ping", "-n", "4", "8.8.8.8"],
    capture_output=True,
    text=True
)

print(result.stdout)


Pinging 8.8.8.8 with 32 bytes of data:
Request timed out.
Request timed out.
Request timed out.
Request timed out.

Ping statistics for 8.8.8.8:
    Packets: Sent = 4, Received = 0, Lost = 4 (100% loss),



In [8]:
import subprocess

ip = "8.8.8.8"
port = 443

command = [
    "powershell",
    "-Command",
    f"Test-NetConnection -ComputerName {ip} -Port {port}"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print(result.stdout)



ComputerName     : 8.8.8.8
RemoteAddress    : 8.8.8.8
RemotePort       : 443
InterfaceAlias   : Ethernet 2
SourceAddress    : 10.0.9.200
TcpTestSucceeded : True






In [9]:
import subprocess

ip = "8.8.8.8"
port = 443

command = (
    f"Test-NetConnection {ip} "
    f"-Port {port} "
    f"-InformationLevel Quiet"
)

result = subprocess.run(
    ["powershell", "-Command", command],
    capture_output=True,
    text=True
)

if result.stdout.strip() == "True":
    print(f"SUCCESS: {ip}:{port} is reachable")
else:
    print(f"FAILED: {ip}:{port} is not reachable")

SUCCESS: 8.8.8.8:443 is reachable


In [10]:
import subprocess

ip = "8.8.8.8"

result = subprocess.run(
    ["ping", "-n", "4", ip],
    capture_output=True,
    text=True
)

print(result.stdout)


Pinging 8.8.8.8 with 32 bytes of data:
Request timed out.
Request timed out.
Request timed out.
Request timed out.

Ping statistics for 8.8.8.8:
    Packets: Sent = 4, Received = 0, Lost = 4 (100% loss),



Yes. Your PowerShell command:

```powershell
tnc -p 443 8.8.8.8
```

is shorthand for:

```powershell
Test-NetConnection 8.8.8.8 -Port 443
```

From Python on Windows, you can execute that PowerShell command using `subprocess`.

### Python example — Test-NetConnection

```python
import subprocess

ip = "8.8.8.8"
port = 443

command = [
    "powershell",
    "-Command",
    f"Test-NetConnection -ComputerName {ip} -Port {port}"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print(result.stdout)
```

You should get output similar to:

```text
ComputerName     : 8.8.8.8
RemoteAddress    : 8.8.8.8
RemotePort       : 443
InterfaceAlias   : Ethernet 2
SourceAddress    : 10.0.9.200
TcpTestSucceeded : True
```

### Better for network automation: return only True/False

If you're writing automation, you usually don't need all the PowerShell output:

```python
import subprocess

ip = "8.8.8.8"
port = 443

command = (
    f"Test-NetConnection {ip} "
    f"-Port {port} "
    f"-InformationLevel Quiet"
)

result = subprocess.run(
    ["powershell", "-Command", command],
    capture_output=True,
    text=True
)

if result.stdout.strip() == "True":
    print(f"SUCCESS: {ip}:{port} is reachable")
else:
    print(f"FAILED: {ip}:{port} is not reachable")
```

Expected:

```text
SUCCESS: 8.8.8.8:443 is reachable
```

One important distinction: **this is not technically a ping**. Your command is testing whether a **TCP connection to port 443** can be established.

```text
ping 8.8.8.8
     │
     └── ICMP reachability test

Test-NetConnection 8.8.8.8 -Port 443
     │
     └── TCP port connectivity test
```

If you specifically want Python to perform the equivalent of:

```powershell
ping 8.8.8.8
```

then you can use:

```python
import subprocess

ip = "8.8.8.8"

result = subprocess.run(
    ["ping", "-n", "4", ip],
    capture_output=True,
    text=True
)

print(result.stdout)
```

For network-engineering automation, I recommend learning the `subprocess` approach first because the same concept can be used to call `ping`, `tracert`, PowerShell networking commands, and other system utilities from Python.


In [11]:
import subprocess
import re

ip = "8.8.8.8"

result = subprocess.run(
    ["ping", "-n", "4", ip],
    capture_output=True,
    text=True
)

output = result.stdout

print(output)

# Packet statistics
packet_match = re.search(
    r"Sent = (\d+), Received = (\d+), Lost = (\d+) \((\d+)% loss\)",
    output
)

if packet_match:
    sent = int(packet_match.group(1))
    received = int(packet_match.group(2))
    lost = int(packet_match.group(3))
    loss_pct = int(packet_match.group(4))

    print(f"Sent: {sent}")
    print(f"Received: {received}")
    print(f"Lost: {lost}")
    print(f"Loss: {loss_pct}%")

# RTT statistics
rtt_match = re.search(
    r"Minimum = (\d+)ms, Maximum = (\d+)ms, Average = (\d+)ms",
    output
)

if rtt_match:
    min_rtt = int(rtt_match.group(1))
    max_rtt = int(rtt_match.group(2))
    avg_rtt = int(rtt_match.group(3))

    print(f"Min RTT: {min_rtt} ms")
    print(f"Max RTT: {max_rtt} ms")
    print(f"Avg RTT: {avg_rtt} ms")


Pinging 8.8.8.8 with 32 bytes of data:
Request timed out.
Request timed out.
Request timed out.
Request timed out.

Ping statistics for 8.8.8.8:
    Packets: Sent = 4, Received = 0, Lost = 4 (100% loss),

Sent: 4
Received: 0
Lost: 4
Loss: 100%


In [12]:
import subprocess
import re

def ping_host(ip):
    result = subprocess.run(
        ["ping", "-n", "4", ip],
        capture_output=True,
        text=True
    )

    output = result.stdout

    stats = {}

    packet = re.search(
        r"Sent = (\d+), Received = (\d+), Lost = (\d+) \((\d+)% loss\)",
        output
    )

    rtt = re.search(
        r"Minimum = (\d+)ms, Maximum = (\d+)ms, Average = (\d+)ms",
        output
    )

    if packet:
        stats["sent"] = int(packet.group(1))
        stats["received"] = int(packet.group(2))
        stats["lost"] = int(packet.group(3))
        stats["loss_percent"] = int(packet.group(4))

    if rtt:
        stats["min_rtt"] = int(rtt.group(1))
        stats["max_rtt"] = int(rtt.group(2))
        stats["avg_rtt"] = int(rtt.group(3))

    return stats

print(ping_host("8.8.8.8"))

{'sent': 4, 'received': 0, 'lost': 4, 'loss_percent': 100}


In [13]:
import subprocess

ip = "8.8.8.8"

result = subprocess.run(
    ["ping", "-n", "1", ip],
    capture_output=True,
    text=True
)

if "TTL=" in result.stdout:
    print(f"{ip} is reachable")
else:
    print(f"{ip} is unreachable")

8.8.8.8 is unreachable


In [14]:
import subprocess

try:
    result = subprocess.run(
        ["ping", "-n", "4", "999.999.999.999"],
        capture_output=True,
        text=True,
        check=True
    )
except subprocess.CalledProcessError as e:
    print("Ping failed")
    print(e)

Ping failed
Command '['ping', '-n', '4', '999.999.999.999']' returned non-zero exit status 1.


In [15]:
import subprocess

ip = "10.255.255.1"

result = subprocess.run(
    ["ping", "-n", "2", ip],
    capture_output=True,
    text=True
)

output = result.stdout

if "Request timed out" in output:
    print(f"{ip} timed out")

elif "Destination host unreachable" in output:
    print(f"{ip} unreachable")

else:
    print(f"{ip} reachable")

10.255.255.1 timed out


In [16]:
import subprocess

try:
    result = subprocess.run(
        ["ping", "-n", "1", "8.8.8.8"],
        capture_output=True,
        text=True
    )

except FileNotFoundError:
    print("ping command not found")

In [17]:
import subprocess

def ping_host(ip):
    try:
        result = subprocess.run(
            ["ping", "-n", "4", ip],
            capture_output=True,
            text=True,
            timeout=10
        )

        output = result.stdout

        if "TTL=" in output:
            return {
                "ip": ip,
                "status": "reachable"
            }

        elif "Request timed out" in output:
            return {
                "ip": ip,
                "status": "timeout"
            }

        elif "Destination host unreachable" in output:
            return {
                "ip": ip,
                "status": "unreachable"
            }

        else:
            return {
                "ip": ip,
                "status": "unknown"
            }

    except subprocess.TimeoutExpired:
        return {
            "ip": ip,
            "status": "command_timeout"
        }

    except FileNotFoundError:
        return {
            "ip": ip,
            "status": "ping_not_found"
        }

    except Exception as e:
        return {
            "ip": ip,
            "status": "error",
            "message": str(e)
        }

print(ping_host("8.8.8.8"))

{'ip': '8.8.8.8', 'status': 'command_timeout'}


When automating network tests, don't assume ping always succeeds. Handle these common error conditions:

1. Invalid IP Address
import subprocess

try:
    result = subprocess.run(
        ["ping", "-n", "4", "999.999.999.999"],
        capture_output=True,
        text=True,
        check=True
    )
except subprocess.CalledProcessError as e:
    print("Ping failed")
    print(e)

2. Ping Timeout / Host Unreachable
import subprocess

ip = "10.255.255.1"

result = subprocess.run(
    ["ping", "-n", "2", ip],
    capture_output=True,
    text=True
)

output = result.stdout

if "Request timed out" in output:
    print(f"{ip} timed out")

elif "Destination host unreachable" in output:
    print(f"{ip} unreachable")

else:
    print(f"{ip} reachable")

3. Command Not Found

If ping.exe is missing or PATH is broken:

import subprocess

try:
    result = subprocess.run(
        ["ping", "-n", "1", "8.8.8.8"],
        capture_output=True,
        text=True
    )

except FileNotFoundError:
    print("ping command not found")

4. Overall Exception Handling
import subprocess

def ping_host(ip):
    try:
        result = subprocess.run(
            ["ping", "-n", "4", ip],
            capture_output=True,
            text=True,
            timeout=10
        )

        output = result.stdout

        if "TTL=" in output:
            return {
                "ip": ip,
                "status": "reachable"
            }

        elif "Request timed out" in output:
            return {
                "ip": ip,
                "status": "timeout"
            }

        elif "Destination host unreachable" in output:
            return {
                "ip": ip,
                "status": "unreachable"
            }

        else:
            return {
                "ip": ip,
                "status": "unknown"
            }

    except subprocess.TimeoutExpired:
        return {
            "ip": ip,
            "status": "command_timeout"
        }

    except FileNotFoundError:
        return {
            "ip": ip,
            "status": "ping_not_found"
        }

    except Exception as e:
        return {
            "ip": ip,
            "status": "error",
            "message": str(e)
        }

print(ping_host("8.8.8.8"))


Example output:

{'ip': '8.8.8.8', 'status': 'reachable'}


or

{'ip': '10.255.255.1', 'status': 'timeout'}

Network Automation Best Practice

For WAN/Lab automation, return structured data instead of printing:

{
    "device": "rwa01.str05",
    "destination": "8.8.8.8",
    "reachable": True,
    "loss_percent": 0,
    "avg_rtt": 12
}


This makes it easy to:

Save results to JSON
Generate reports
Fail a test case if packet loss > 0%
Compare latency before/after configuration changes
Integrate with pytest or automation pipelines

This is the approach commonly used in health-check and validation scripts.